In [1]:
from structools import *
from aminotools import *
from seqtools import *


In [20]:
from Bio.PDB.ResidueDepth import get_surface, ResidueDepth
from collections import Counter
import pandas as pd

In [3]:
def get_surface_residues(structure, threshold=5, include_distance=False, include_resname=False):
    surface_matrix = GetRDSurface(structure[0])  # Get Residue Depth's Surface Model
    residues = get_residues(structure)           # Get Structure's Residues
    surface = []
    for res in residues:
        dist = min_dist(res.center_of_mass(), surface_matrix)
        if dist <= threshold: 
            res_id = f"{res.resname}{res.id[1]}{res.parent.id}"
            if not include_resname:  res_id = res_id.replace(res.resname, "")
            if include_distance: res_id = [res_id, dist]
            surface.append(res_id)  
    return surface

In [4]:
structure = get_structure("7uv0.pdb")

## Residuos del sitio activo

In [5]:
# Obtener residuos a 6 Å del ligando Ioduro
site = get_active_site(structure, ligands_names=["IOD"], threshold=6)
print(site)

{'703A': ['72A', '90A', '293A']}


### Entonces 703 es el residuo que se une al ligando Ioduro y el 72A, 90A y 293A se encuentran a una distancia de 6A

In [6]:
# Obtener residuos a 6 Å del ligando Sodio 
site = get_active_site(structure, ligands_names=["NA"], threshold=6)
print(site)

{'702A': ['67A', '69A', '71A', '72A', '144A'], '701A': ['72A', '413A', '416A', '417A']}


### Idem, pero acá hay dos ligandos de sodio 

## Residuos de superficie

In [7]:
surface_residues = get_surface_residues(structure, threshold=5)
print(surface_residues)

/home/fran/.local/lib/python3.10/site-packages/Bio/PDB/ResidueDepth.py:490: BiopythonWarning: I:IOD not in radii library.
  warnings.warn(f"{at_name}:{resname} not in radii library.", BiopythonWarning)


['10A', '11A', '12A', '13A', '14A', '15A', '16A', '17A', '18A', '19A', '20A', '21A', '22A', '23A', '24A', '25A', '26A', '27A', '28A', '29A', '30A', '31A', '32A', '33A', '34A', '35A', '36A', '56A', '57A', '58A', '59A', '60A', '61A', '62A', '63A', '65A', '66A', '67A', '68A', '69A', '74A', '75A', '78A', '79A', '81A', '82A', '83A', '85A', '86A', '88A', '89A', '91A', '92A', '93A', '95A', '96A', '98A', '99A', '100A', '102A', '103A', '104A', '105A', '106A', '107A', '108A', '110A', '111A', '112A', '113A', '114A', '115A', '116A', '119A', '122A', '123A', '124A', '125A', '126A', '127A', '128A', '129A', '130A', '131A', '132A', '133A', '134A', '135A', '136A', '137A', '138A', '139A', '140A', '141A', '142A', '144A', '145A', '147A', '148A', '149A', '150A', '154A', '156A', '157A', '158A', '159A', '160A', '161A', '162A', '163A', '164A', '165A', '166A', '167A', '168A', '169A', '170A', '171A', '172A', '173A', '174A', '175A', '176A', '177A', '178A', '179A', '180A', '181A', '182A', '183A', '190A', '191A', '

## Residuos del core

In [9]:
# Obtener todos los residuos con nombre + número + cadena (ej: 'GLU45A')
all_residues = [f"{res.resname}{res.id[1]}{res.parent.id}" for res in get_residues(structure)]

# Superficie
surface_residues = get_surface_residues(structure, threshold=5, include_resname=True)

# Sitio activo alrededor de IOD y NA
site_IOD = get_active_site(structure, ligands_names=["IOD"], threshold=6, include_resname=True)
site_NA = get_active_site(structure, ligands_names=["NA"], threshold=6, include_resname=True)

# Unir todos los residuos del sitio activo en una sola lista
active_site_residues = []
for lst in site_IOD.values():
    active_site_residues.extend(lst)
for lst in site_NA.values():
    active_site_residues.extend(lst)

# Convertir a conjuntos
set_all = set(all_residues)
set_surface = set(surface_residues)
set_active = set(active_site_residues)

# Obtener el core: todos menos superficie y sitio activo
core_residues = list(set_all - set_surface - set_active)

# Resultado
print("Residuos del core (núcleo):")
print(core_residues)

Residuos del core (núcleo):
['SER257A', 'TYR259A', 'ALA151A', 'VAL266A', 'THR117A', 'LEU282A', 'MET325A', 'ALA296A', 'ALA278A', 'VAL76A', 'LEU121A', 'VAL401A', 'TYR118A', 'GLN94A', 'VAL445A', 'PHE87A', 'PRO152A', 'PHE109A', 'ASN97A', 'LEU524A', 'TYR242A', 'ALA80A', 'PHE431A', 'MET435A', 'GLY425A', 'VAL419A', 'VAL73A', 'SER456A', 'LEU344A', 'TYR269A', 'LEU143A', 'GLY527A', 'ALA64A', 'VAL261A', 'PRO426A', 'ALA430A', 'ALA153A', 'ALA70A', 'GLY84A', 'TYR120A', 'LEU427A', 'PHE202A', 'PRO326A', 'PRO77A', 'PHE247A', 'LEU289A', 'GLY421A', 'SER424A', 'TRP255A', 'VAL254A', 'THR534A', 'THR101A', 'GLY146A', 'LEU352A', 'ILE155A', 'GLY260A', 'GLY251A', 'LEU328A']


In [18]:
# 1. Todos los residuos
all_residues = [f"{res.resname}{res.id[1]}{res.parent.id}" for res in get_residues(structure)]

# 2. Superficie
surface_residues = get_surface_residues(structure, threshold=6, include_resname=True)

# 3. Sitio activo (IOD y NA)
site_IOD = get_active_site(structure, ligands_names=["IOD"], threshold=6, include_resname=True)
site_NA = get_active_site(structure, ligands_names=["NA"], threshold=6, include_resname=True)

# Unir todos los residuos del sitio activo
active_site_residues = []
for lst in site_IOD.values():
    active_site_residues.extend(lst)
for lst in site_NA.values():
    active_site_residues.extend(lst)

# Convertir a sets (elimina duplicados automágicamente)
set_all = set(all_residues)
set_surface = set(surface_residues)
set_active = set(active_site_residues)

# El core: los que no están en superficie ni en el sitio activo
set_core = set_all - set_surface - set_active

# 🧮 Conteo
print(f"Total de residuos     : {len(set_all)}")
print(f"Residuos de superficie: {len(set_surface)}")
print(f"Residuos en sitio activo (IOD y NA): {len(set_active)}")
print(f"Residuos del core     : {len(set_core)}")

# Verificación
total_clasificados = len(set_surface | set_active | set_core)
print(f"\n¿Coincide con el total?: {total_clasificados == len(set_all)}")

Total de residuos     : 501
Residuos de superficie: 474
Residuos en sitio activo (IOD y NA): 10
Residuos del core     : 25

¿Coincide con el total?: True


# Armar dataframe

In [21]:
# 5. Crear listas etiquetadas
data = []

for res in surface_residues:
    data.append((res, "superficie"))
for res in active_site_residues:
    data.append((res, "sitio activo"))
for res in core_residues:
    data.append((res, "core"))

# 6. Crear DataFrame
df = pd.DataFrame(data, columns=["Residuo", "Categoría"])

# Opcional: ordenar por cadena/número
df = df.sort_values("Residuo").reset_index(drop=True)

# Mostrar resumen
print(df.head())
print("\nConteo por categoría:")
print(df["Categoría"].value_counts())

   Residuo   Categoría
0  ALA102A  superficie
1   ALA10A  superficie
2  ALA128A  superficie
3  ALA140A  superficie
4   ALA14A  superficie

Conteo por categoría:
Categoría
superficie      474
core             58
sitio activo     12
Name: count, dtype: int64


In [22]:
df.to_csv("residuos_clasificados.csv", index=False)